# 🚀 FATFORMER-XLA: KHỞI TẠO CẤU TRÚC DRIVE 5TB & TỰ ĐỘNG NẠP DỮ LIỆU
> **Dự án**: FatFormer-XLA (CVPR 2024 Re-implementation & Hardening)  
> **Mục tiêu**: Chuẩn bị đầy đủ 100% kho dữ liệu đóng gói nguyên khối `.tar` và bộ trọng số trên Google Drive 5TB.  
> **Tương thích**: Google Colab (Standard & Pro / High-RAM), Tốc độ mạng Cloud 100 – 300 MB/s.

---
### 📋 Quy trình nạp dữ liệu chuẩn hóa:
1. **Bước 1**: Kết nối Google Drive & Cài đặt công cụ tải đa luồng (`aria2c`, `p7zip-full`).
2. **Bước 2**: Xác nhận / Tải bộ Pretrained Weights (`ViT-L-14.pt` & `fatformer_4class_ckpt.pth`).
3. **Bước 3**: Tải, giải nén & đóng gói tập Validation 20 lớp (`progan_val.tar` ~ 792 MB).
4. **Bước 4**: Tải 7 phần, trích xuất lọc trực tiếp 4 lớp & đóng gói tập Train (`progan_train.tar` ~ 4.8 GB - **Chống tràn bộ nhớ Colab**).
5. **Bước 5**: Tải & đóng gói 8 họ GANs Benchmark (`test_benchmark_gans.tar` ~ 12.5 GB / 18.68 GB zip gốc).
6. **Bước 6**: **Tự động tải & đóng gói 10 mô hình Diffusion (`test_benchmark_diffusion.tar` ~ 6.5 GB) & GenImage Staging (`diffusion_staging.tar` ~ 720 MB)**.
7. **Bước 7**: Báo cáo kiểm kê toàn diện Google Drive & Đánh giá trạng thái sẵn sàng.
8. **Bước 8**: Kiểm tra tính toàn vẹn (Smoke Test header) của toàn bộ các file `.tar`.


## 1. Kết nối Google Drive & Thiết lập Môi trường Tải Siêu Tốc
Thực hiện Mount Google Drive vào đường dẫn chuẩn `/content/drive`.  
Hệ thống sẽ tự động khởi tạo 4 thư mục tiêu chuẩn:
* `MyDrive/Fatformer/pretrained/`
* `MyDrive/Fatformer/checkpoint/`
* `MyDrive/Fatformer/log/`
* `MyDrive/Fatformer/datasets/`


In [ ]:
# 1. Mount Google Drive & Khởi tạo thư mục
from google.colab import drive
import os
import shutil
import time

print("🔗 Đang kết nối Google Drive...")
drive.mount('/content/drive')

# Đường dẫn thư mục gốc FatFormer trên Drive
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
DIR_CHECKPOINT = os.path.join(DRIVE_ROOT, "checkpoint")
DIR_DATASETS   = os.path.join(DRIVE_ROOT, "datasets")
DIR_LOG        = os.path.join(DRIVE_ROOT, "log")
DIR_PRETRAINED = os.path.join(DRIVE_ROOT, "pretrained")

# Tự động tạo cây thư mục nếu chưa tồn tại
for d in [DIR_CHECKPOINT, DIR_DATASETS, DIR_LOG, DIR_PRETRAINED]:
    os.makedirs(d, exist_ok=True)
    print(f"📁 Thư mục sẵn sàng: {d}")

# Kiểm tra dung lượng còn trống trên Google Drive
statvfs = os.statvfs(DRIVE_ROOT)
free_gb = (statvfs.f_frsize * statvfs.f_bavail) / (1024**3)
print(f"💾 Dung lượng Google Drive khả dụng: {free_gb:.2f} GB")

# Cài đặt công cụ aria2 và p7zip-full (chạy êm không in rác)
print()
print("🛠️ Đang cài đặt aria2c và p7zip-full...")
os.system("apt-get install -y -qq aria2 p7zip-full > /dev/null 2>&1")

print()
print("✅ BƯỚC 1 HOÀN TẤT: Đã kết nối Google Drive và sẵn sàng môi trường tải!")


## 2. Kiểm tra & Tải Trọng số Tiền huấn luyện (`pretrained`)
Xác nhận sự hiện diện của 2 tệp trọng số quan trọng:
1. `ViT-L-14.pt` (OpenAI CLIP Backbone ~ 903 MB).
2. `fatformer_4class_ckpt.pth` (Checkpoint chính thức CVPR 2024 ~ 354 MB).
*(Nếu thiếu `ViT-L-14.pt`, script sẽ tự động tải trực tiếp từ máy chủ Microsoft Azure của OpenAI)*


In [ ]:
# 2. Kiểm tra bộ Pretrained Weights
vit_path = os.path.join(DIR_PRETRAINED, "ViT-L-14.pt")
ckpt_path = os.path.join(DIR_PRETRAINED, "fatformer_4class_ckpt.pth")

def check_file(path, expected_mb_min=100):
    if os.path.exists(path) and os.path.getsize(path) >= expected_mb_min * 1024 * 1024:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✅ ĐÃ CÓ: {os.path.basename(path)} ({size_mb:.2f} MB)")
        return True
    else:
        print(f"  ❌ CHƯA CÓ: {os.path.basename(path)}")
        return False

print("🔍 Đang kiểm tra thư mục pretrained trên Drive:")
has_vit = check_file(vit_path, expected_mb_min=800)
has_ckpt = check_file(ckpt_path, expected_mb_min=300)

if not has_vit:
    print()
    print("⬇️ Đang tải tự động ViT-L-14.pt từ OpenAI Azure CDN...")
    os.system('aria2c -x 16 -s 16 -o "ViT-L-14.pt" https://openaipublic.azureedge.net/clip/models/b8cca3fd8d72c9937d9c052a7f6bb60474e327045934442a604cd7e06b774b9d/ViT-L-14.pt')
    if os.path.exists("ViT-L-14.pt"):
        shutil.move("ViT-L-14.pt", os.path.join(DIR_PRETRAINED, "ViT-L-14.pt"))
        print(f"  ✅ Đã tải và lưu ViT-L-14.pt vào: {DIR_PRETRAINED}")

if not has_ckpt:
    print()
    print("⚠️ Lưu ý: fatformer_4class_ckpt.pth chưa có trên Drive.")
    print("   Bạn vui lòng tải file này từ Baidu (pass: 9i5w) và upload trực tiếp vào:")
    print(f"   📂 {DIR_PRETRAINED}")
else:
    print()
    print("✅ BƯỚC 2 HOÀN TẤT: Toàn bộ bộ Pretrained Weights đã sẵn sàng 100%!")


## 3. Tải & Đóng gói Tập Validation (`progan_val.tar`)
* **Nguồn**: Hugging Face `sywang/CNNDetection` (`progan_val.zip` ~ 792 MB).
* **Nội dung**: 20 danh mục (8.000 ảnh: 4.000 Real + 4.000 Fake).
* **Cơ chế an toàn**: Tự động làm phẳng thư mục con (nếu zip chứa thư mục lồng), đếm số lượng lớp đối tượng trước khi tar $
ightarrow$ **Đảm bảo không bao giờ sinh ra file tar 0.01 MB**.


In [ ]:
# 3. Tải & Đóng gói tập Validation
os.chdir("/content")
target_val_tar = os.path.join(DIR_DATASETS, "progan_val.tar")

# Kiểm tra nếu file hợp lệ (> 500 MB) đã tồn tại trên Drive
if os.path.exists(target_val_tar) and os.path.getsize(target_val_tar) > 500 * 1024 * 1024:
    print(f"✅ File {target_val_tar} đã tồn tại hợp lệ ({os.path.getsize(target_val_tar)/(1024**2):.2f} MB). Bỏ qua bước tải.")
else:
    print("⬇️ Đang tải progan_val.zip từ Hugging Face qua aria2c (có gắn cờ -o)...")
    if os.path.exists("/content/progan_val.zip"):
        os.remove("/content/progan_val.zip")
    os.system('aria2c -x 16 -s 16 -o "progan_val.zip" "https://huggingface.co/datasets/sywang/CNNDetection/resolve/main/progan_val.zip"')

    print("📦 Đang giải nén tập validation...")
    shutil.rmtree("/content/val", ignore_errors=True)
    os.makedirs("/content/val", exist_ok=True)
    os.system('unzip -q /content/progan_val.zip -d /content/val')
    if os.path.exists("/content/progan_val.zip"):
        os.remove("/content/progan_val.zip")

    # Làm phẳng nếu có thư mục lồng /content/val/val
    inner_val = "/content/val/val"
    if os.path.exists(inner_val) and os.path.isdir(inner_val):
        for item in os.listdir(inner_val):
            shutil.move(os.path.join(inner_val, item), "/content/val")
        os.rmdir(inner_val)

    # Kiểm tra xác nhận đủ danh mục và ảnh
    val_classes = [d for d in os.listdir("/content/val") if os.path.isdir(os.path.join("/content/val", d))]
    print(f"🔍 Kiểm tra dữ liệu: Tìm thấy {len(val_classes)}/20 danh mục trong /content/val")
    assert len(val_classes) >= 15, f"❌ Lỗi: Thư mục val không đủ số lượng danh mục (chỉ có {len(val_classes)})!"

    print(f"🚀 Đang đóng gói progan_val.tar lưu sang Google Drive: {target_val_tar}...")
    os.system(f'tar -cf "{target_val_tar}" -C /content val')
    shutil.rmtree("/content/val", ignore_errors=True)

    final_size_mb = os.path.getsize(target_val_tar) / (1024**2)
    assert final_size_mb > 500, f"❌ Lỗi: Kích thước file tar quá nhỏ ({final_size_mb:.2f} MB), nghi vấn bị rỗng!"
    print(f"✅ BƯỚC 3 HOÀN TẤT: progan_val.tar đã được cất an toàn tại Drive ({final_size_mb:.2f} MB)!")


## 4. Tải Tập Huấn Luyện ProGAN 4-Class (`progan_train.tar`)
* **Thách thức lớn**: Kho Hugging Face chứa 20 lớp đầy đủ (~70 GB trong 7 file 7z). Nếu giải nén cả 70 GB ảnh trên Colab sẽ bị **Tràn ổ đĩa (Disk Full Crash)**.
* **Giải pháp Đột Phá**:
  1. Tải 7 file song song với danh sách gán cứng tên file `out=progan_train.7z.00x`.
  2. Dùng lệnh `7z x -y progan_train.7z.001` nối tiếp và xóa ngay 70 GB file nén sau khi giải nén.
  3. **Chỉ trích xuất lọc trực tiếp 4 lớp chuẩn CVPR 2024**: `car`, `cat`, `chair`, `horse` (24.000 ảnh ~ 4.8 GB) bằng cú pháp lọc của `unzip`.
  4. Kiểm tra xác nhận đủ ảnh trong `0_real` và `1_fake` trước khi đóng gói $
ightarrow$ Không thể sinh file rỗng.
* **Thời gian ước tính**: ~5 – 7 phút.


In [ ]:
# 4. Tải, Lọc 4 Lớp & Đóng Gói Tập Train (Chống tràn bộ nhớ Colab)
os.chdir("/content")
target_train_tar = os.path.join(DIR_DATASETS, "progan_train.tar")

# Chỉ bỏ qua nếu file đã tồn tại VÀ dung lượng > 3.5 GB (tránh file lỗi/rỗng)
if os.path.exists(target_train_tar) and os.path.getsize(target_train_tar) > 3.5 * 1024 * 1024 * 1024:
    print(f"✅ File {target_train_tar} đã tồn tại ({os.path.getsize(target_train_tar)/(1024**3):.2f} GB). Bỏ qua bước tải.")
else:
    # 1. Chuẩn bị file danh sách URL với out cố định
    with open("urls_train.txt", "w") as f:
        for i in range(1, 8):
            fname = f"progan_train.7z.00{i}"
            url = f"https://huggingface.co/datasets/sywang/CNNDetection/resolve/main/{fname}"
            f.write(url + "\n  out=" + fname + "\n")

    print("⬇️ Đang tải song song 7 phần của tập Train (tổng ~70 GB)...")
    os.system("aria2c -x 16 -s 16 -j 7 -i urls_train.txt")
    if os.path.exists("urls_train.txt"):
        os.remove("urls_train.txt")

    print()
    print("📦 Đang giải nén nối tiếp các phần 7z...")
    os.system("7z x -y progan_train.7z.001")
    os.system("rm -f progan_train.7z.*")

    # 2. Trích xuất CHỌN LỌC đúng 4 lớp chuẩn: car, cat, chair, horse
    shutil.rmtree("/content/train", ignore_errors=True)
    shutil.rmtree("/content/train_raw", ignore_errors=True)
    os.makedirs("/content/train", exist_ok=True)

    if os.path.exists("/content/progan_train.zip"):
        print("📦 Đang trích xuất lọc trực tiếp 4 lớp (car, cat, chair, horse) từ zip...")
        os.system('unzip -q /content/progan_train.zip "car/*" "cat/*" "chair/*" "horse/*" "*/car/*" "*/cat/*" "*/chair/*" "*/horse/*" -d /content/train_raw/')
        if os.path.exists("/content/progan_train.zip"):
            os.remove("/content/progan_train.zip")
        
        # Di chuyển vào /content/train
        for root, dirs, files in os.walk("/content/train_raw"):
            for cls in ["car", "cat", "chair", "horse"]:
                if cls in dirs:
                    src_cls = os.path.join(root, cls)
                    dst_cls = os.path.join("/content/train", cls)
                    if not os.path.exists(dst_cls):
                        shutil.move(src_cls, dst_cls)
        shutil.rmtree("/content/train_raw", ignore_errors=True)
    else:
        # Nếu 7z giải nén trực tiếp ra thư mục ảnh
        for cls in ["car", "cat", "chair", "horse"]:
            for cand in [f"/content/progan_train/{cls}", f"/content/{cls}"]:
                if os.path.exists(cand):
                    dst_cls = os.path.join("/content/train", cls)
                    if not os.path.exists(dst_cls):
                        shutil.move(cand, dst_cls)
        shutil.rmtree("/content/progan_train", ignore_errors=True)

    # 3. Kiểm tra xác thực nghiêm ngặt
    target_classes = ["car", "cat", "chair", "horse"]
    for c in target_classes:
        r_dir = f"/content/train/{c}/0_real"
        f_dir = f"/content/train/{c}/1_fake"
        assert os.path.isdir(r_dir) and len(os.listdir(r_dir)) > 100, f"❌ Thiếu ảnh 0_real tại: {r_dir}"
        assert os.path.isdir(f_dir) and len(os.listdir(f_dir)) > 100, f"❌ Thiếu ảnh 1_fake tại: {f_dir}"
        print(f"  ✅ Lớp {c}: {len(os.listdir(r_dir))} ảnh thật, {len(os.listdir(f_dir))} ảnh giả.")

    # 4. Đóng gói lưu vào Google Drive
    print()
    print(f"🚀 Đang đóng gói progan_train.tar sang Drive: {target_train_tar}...")
    os.system(f'tar -cf "{target_train_tar}" -C /content train')
    shutil.rmtree("/content/train", ignore_errors=True)

    final_size_gb = os.path.getsize(target_train_tar) / (1024**3)
    assert final_size_gb > 3.0, f"❌ Lỗi: Kích thước file train quá nhỏ ({final_size_gb:.2f} GB)!"
    print(f"✅ BƯỚC 4 HOÀN TẤT: progan_train.tar đã lưu thành công ({final_size_gb:.2f} GB)!")


## 5. Tải Tập Kiểm Thử 8 Họ GANs Benchmark (`test_benchmark_gans.tar`)
* **Nguồn**: `CNN_synth_testset.zip` (~18.68 GB) từ CNNDetection.
* **Mô hình bao gồm**: ProGAN, StyleGAN, StyleGAN2, BigGAN, CycleGAN, StarGAN, GauGAN, Deepfake.
* **Tính năng thông minh**: Nếu file 18.68 GB đã vô tình tải về dưới tên mã hash lạ của GCP CDN, script sẽ tự động tìm kiếm, đổi tên và tái sử dụng ngay lập tức mà không cần tải lại!


In [ ]:
# 5. Tải & Đóng Gói 8 Họ GANs Testset
os.chdir("/content")
target_gans_tar = os.path.join(DIR_DATASETS, "test_benchmark_gans.tar")

# Chỉ bỏ qua nếu file đã tồn tại VÀ dung lượng > 8 GB (tránh file lỗi/rỗng)
if os.path.exists(target_gans_tar) and os.path.getsize(target_gans_tar) > 8 * 1024 * 1024 * 1024:
    print(f"✅ File {target_gans_tar} đã tồn tại ({os.path.getsize(target_gans_tar)/(1024**3):.2f} GB). Bỏ qua bước tải.")
else:
    # Tự động tìm kiếm file 18.68 GB tải dở dưới tên mã hash CDN
    zip_found = False
    for f in os.listdir("/content"):
        fp = os.path.join("/content", f)
        # 18.68 GB ~ 17.4 GiB (ngưỡng 16.5 GiB)
        if os.path.isfile(fp) and os.path.getsize(fp) > 16.5 * 1024 * 1024 * 1024:
            print(f"🎯 Phát hiện file dữ liệu lớn 18.68 GB đã tải sẵn tại: {f}")
            shutil.move(fp, "/content/CNN_synth_testset.zip")
            zip_found = True
            break

    if not zip_found and not os.path.exists("/content/CNN_synth_testset.zip"):
        print("⬇️ Đang tải CNN_synth_testset.zip (18.68 GB) với cờ đặt tên file -o...")
        os.system('aria2c -x 16 -s 16 -o "CNN_synth_testset.zip" "https://huggingface.co/datasets/sywang/CNNDetection/resolve/main/CNN_synth_testset.zip"')

    print("📦 Đang giải nén tập test GANs...")
    shutil.rmtree("/content/test", ignore_errors=True)
    os.makedirs("/content/test", exist_ok=True)
    os.system('unzip -q /content/CNN_synth_testset.zip -d /content/test')
    if os.path.exists("/content/CNN_synth_testset.zip"):
        os.remove("/content/CNN_synth_testset.zip")

    # Tự động làm phẳng nếu file zip giải nén ra thư mục con CNN_synth_testset
    for inner in ["CNN_synth_testset/test", "CNN_synth_testset"]:
        ip = os.path.join("/content/test", inner)
        if os.path.exists(ip) and os.path.isdir(ip):
            print(f"🔧 Đang sắp xếp lại cấu trúc thư mục từ {ip}...")
            for sub_name in os.listdir(ip):
                src = os.path.join(ip, sub_name)
                dst = os.path.join("/content/test", sub_name)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join("/content/test", inner.split("/")[0]), ignore_errors=True)

    # Kiểm tra xác nhận sự hiện diện của các họ GANs
    expected_gans = ["progan", "stylegan", "stylegan2", "biggan", "cyclegan", "stargan", "gaugan", "deepfake"]
    found_gans = [g for g in expected_gans if os.path.exists(os.path.join("/content/test", g))]
    print(f"🔍 Đã xác nhận {len(found_gans)}/{len(expected_gans)} họ GANs trong /content/test: {found_gans}")
    assert len(found_gans) >= 4, f"❌ Lỗi: Thư mục test không chứa các họ GANs hợp lệ!"

    print(f"🚀 Đang đóng gói sang Drive: {target_gans_tar}...")
    os.system(f'tar -cf "{target_gans_tar}" -C /content test')
    shutil.rmtree("/content/test", ignore_errors=True)

    final_size = os.path.getsize(target_gans_tar) / (1024**3)
    assert final_size > 8.0, f"❌ Lỗi: File test_benchmark_gans quá nhỏ ({final_size:.2f} GB)!"
    print(f"✅ BƯỚC 5 HOÀN TẤT: test_benchmark_gans.tar đã lưu thành công ({final_size:.2f} GB)!")


## 6. Tự Động Tải & Đóng Gói Bộ Diffusion Models & GenImage Staging
Thực hiện tự động hóa 100% việc tạo 2 tệp `.tar`:
1. **`test_benchmark_diffusion.tar` (~6.5 GB)**: Tải trực tiếp từ OneDrive chính thức của tác giả FatFormer bằng `onedrivedownloader`, giải nén 10 mô hình Diffusion (`guided`, `ldm_200`, `glide_...`, v.v.) và đóng gói nguyên khối sang Drive.
2. **`diffusion_staging.tar` (~720 MB)**: Tải và trích xuất chuẩn xác **3.600 ảnh** (1.200 SD 1.5 Real/Fake + 600 Midjourney Real/Fake) từ Hugging Face `Tiny-GenImage` làm chất xúc tác cho Chiến lược 3 (SRM Hardening).


In [ ]:
# 6. TỰ ĐỘNG TẢI & ĐÓNG GÓI BỘ DIFFUSION BENCHMARK VÀ GENIMAGE STAGING SANG DRIVE
import os
import shutil
import time

target_diff_tar = os.path.join(DIR_DATASETS, "test_benchmark_diffusion.tar")
target_staging_tar = os.path.join(DIR_DATASETS, "diffusion_staging.tar")

# 1. Cài đặt các thư viện cần thiết (chạy siêu tốc trong ~5 giây trên Colab)
print("🛠️ Đang cài đặt onedrivedownloader & datasets...")
os.system("pip install -q onedrivedownloader datasets pillow")

# =========================================================================
# PHẦN 6.1: TỰ ĐỘNG TẢI & ĐÓNG GÓI test_benchmark_diffusion.tar (~6.5 GB)
# =========================================================================
print()
print("=" * 70)
print("📦 [1/2] XỬ LÝ TEST_BENCHMARK_DIFFUSION.TAR (10 HỌ DIFFUSION BENCHMARK)")
print("=" * 70)

if os.path.exists(target_diff_tar) and os.path.getsize(target_diff_tar) > 1 * 1024 * 1024 * 1024:
    size_gb = os.path.getsize(target_diff_tar) / (1024**3)
    print(f"✅ ĐÃ CÓ: test_benchmark_diffusion.tar trên Drive ({size_gb:.2f} GB). Bỏ qua bước tải.")
else:
    from onedrivedownloader import download

    onedrive_url = "https://1drv.ms/u/s!Aqkrc9gPuk8jqaM1LAthli7KdRhr2A?e=ebbG9r"
    diff_zip_path = "/content/diffusion_test.zip"
    extract_dir = "/content/diffusion_test"

    print("⬇️ Đang tải trực tiếp gói Diffusion Benchmark từ OneDrive của tác giả FatFormer...")
    download(onedrive_url, filename=diff_zip_path, unzip=False, clean=False)

    print("📦 Đang giải nén tập diffusion test...")
    shutil.rmtree(extract_dir, ignore_errors=True)
    os.makedirs(extract_dir, exist_ok=True)
    os.system(f'unzip -q "{diff_zip_path}" -d "{extract_dir}"')
    if os.path.exists(diff_zip_path):
        os.remove(diff_zip_path)

    # Tự động làm phẳng nếu file zip giải nén ra thư mục con lồng nhau
    for inner in ["diffusion_test", "diffusion", "test"]:
        inner_p = os.path.join(extract_dir, inner)
        if os.path.exists(inner_p) and os.path.isdir(inner_p):
            print(f"🔧 Đang sắp xếp lại cấu trúc thư mục từ {inner_p}...")
            for sub_name in os.listdir(inner_p):
                src = os.path.join(inner_p, sub_name)
                dst = os.path.join(extract_dir, sub_name)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(inner_p, ignore_errors=True)

    # Kiểm tra xác nhận các mô hình Diffusion
    found_subsets = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]
    print(f"🔍 Tìm thấy {len(found_subsets)} thư mục kiểm thử Diffusion: {found_subsets}")
    assert len(found_subsets) >= 3, "❌ Lỗi: Thư mục diffusion_test không chứa đủ dữ liệu!"

    print(f"🚀 Đang đóng gói test_benchmark_diffusion.tar sang Drive: {target_diff_tar}...")
    os.system(f'tar -cf "{target_diff_tar}" -C /content diffusion_test')
    shutil.rmtree(extract_dir, ignore_errors=True)

    final_size_gb = os.path.getsize(target_diff_tar) / (1024**3)
    assert final_size_gb > 1.0, f"❌ Lỗi: File test_benchmark_diffusion quá nhỏ ({final_size_gb:.2f} GB)!"
    print(f"✅ HOÀN TẤT PHẦN 6.1: test_benchmark_diffusion.tar đã lưu thành công ({final_size_gb:.2f} GB)!")

# =========================================================================
# PHẦN 6.2: TỰ ĐỘNG TRÍCH XUẤT & ĐÓNG GÓI diffusion_staging.tar (~720 MB)
# =========================================================================
print()
print("=" * 70)
print("📦 [2/2] XỬ LÝ DIFFUSION_STAGING.TAR (3.600 ẢNH SD 1.5 + MIDJOURNEY V5)")
print("=" * 70)

# Nếu file cũ bị lỗi rỗng (<= 1 MB) thì xóa ngay để nạp lại chuẩn xác
if os.path.exists(target_staging_tar) and os.path.getsize(target_staging_tar) <= 10 * 1024 * 1024:
    print(f"🧹 Đang xóa file cũ bị rỗng: {target_staging_tar}")
    os.remove(target_staging_tar)

if os.path.exists(target_staging_tar) and os.path.getsize(target_staging_tar) > 200 * 1024 * 1024:
    size_mb = os.path.getsize(target_staging_tar) / (1024**2)
    print(f"✅ ĐÃ CÓ: diffusion_staging.tar trên Drive ({size_mb:.2f} MB). Bỏ qua bước trích xuất.")
else:
    from datasets import load_dataset

    staging_dir = "/content/diffusion_staging"
    sd15_real_dir = os.path.join(staging_dir, "stable_diffusion_v_1_5", "0_real")
    sd15_fake_dir = os.path.join(staging_dir, "stable_diffusion_v_1_5", "1_fake")
    mj_real_dir = os.path.join(staging_dir, "midjourney", "0_real")
    mj_fake_dir = os.path.join(staging_dir, "midjourney", "1_fake")

    shutil.rmtree(staging_dir, ignore_errors=True)
    for d in [sd15_real_dir, sd15_fake_dir, mj_real_dir, mj_fake_dir]:
        os.makedirs(d, exist_ok=True)

    print("⬇️ Đang tải tập mẫu GenImage chuẩn từ Hugging Face (TheKernel01/Tiny-GenImage)...")
    try:
        ds = load_dataset("TheKernel01/Tiny-GenImage", split="train")

        # Lấy danh sách tên generator từ ClassLabel schema
        gen_names = ds.features["generator"].names if hasattr(ds.features["generator"], "names") else []
        print(f"📋 Danh sách generators trong dataset: {gen_names}")

        print("🔍 Đang trích xuất 3.600 ảnh (1.200 SD 1.5 Real/Fake + 600 Midjourney Real/Fake)...")
        target_counts = {
            "sd15_real": 1200,
            "sd15_fake": 1200,
            "mj_real": 600,
            "mj_fake": 600,
        }
        counts = {"sd15_real": 0, "sd15_fake": 0, "mj_real": 0, "mj_fake": 0}

        for idx, item in enumerate(ds):
            raw_gen = item["generator"]
            if isinstance(raw_gen, int) and gen_names and raw_gen < len(gen_names):
                gen_name = gen_names[raw_gen]
            else:
                gen_name = str(raw_gen)

            label = int(item["label"])  # 0: real, 1: fake

            # 1. Ảnh THẬT (ImageNet gốc)
            if label == 0 or raw_gen == 0 or gen_name.lower() == "real":
                if counts["sd15_real"] < target_counts["sd15_real"]:
                    c_idx = counts["sd15_real"]
                    item["image"].save(os.path.join(sd15_real_dir, f"{c_idx:05d}.png"))
                    counts["sd15_real"] += 1
                elif counts["mj_real"] < target_counts["mj_real"]:
                    c_idx = counts["mj_real"]
                    item["image"].save(os.path.join(mj_real_dir, f"{c_idx:05d}.png"))
                    counts["mj_real"] += 1

            # 2. Ảnh GIẢ (AI Generated)
            elif label == 1:
                # SD 1.5 (index 6 hoặc tên chứa 'sd15')
                if (raw_gen == 6 or "sd15" in gen_name.lower()) and counts["sd15_fake"] < target_counts["sd15_fake"]:
                    c_idx = counts["sd15_fake"]
                    item["image"].save(os.path.join(sd15_fake_dir, f"{c_idx:05d}.png"))
                    counts["sd15_fake"] += 1
                # Midjourney (index 4 hoặc tên chứa 'midjourney')
                elif (raw_gen == 4 or "midjourney" in gen_name.lower()) and counts["mj_fake"] < target_counts["mj_fake"]:
                    c_idx = counts["mj_fake"]
                    item["image"].save(os.path.join(mj_fake_dir, f"{c_idx:05d}.png"))
                    counts["mj_fake"] += 1

            if idx % 1000 == 0 and idx > 0:
                print(f"  ... Đã duyệt {idx} mẫu | SD15: {counts['sd15_real']}/1200 Real, {counts['sd15_fake']}/1200 Fake | MJ: {counts['mj_real']}/600 Real, {counts['mj_fake']}/600 Fake")

            if all(counts[k] >= target_counts[k] for k in target_counts):
                print(f"  🎯 Đã gom đủ 3.600 ảnh tại vị trí index {idx}!")
                break

        print(f"  ✅ Đã trích xuất hoàn tất:")
        print(f"     - SD 1.5: {counts['sd15_real']} Real, {counts['sd15_fake']} Fake")
        print(f"     - Midjourney: {counts['mj_real']} Real, {counts['mj_fake']} Fake")
        total_extracted = sum(counts.values())
        assert total_extracted == 3600, f"❌ Lỗi: Chỉ trích xuất được {total_extracted}/3.600 ảnh!"

    except Exception as e:
        print(f"⚠️ Ngoại lệ khi tải/trích xuất từ Hugging Face: {e}")
        raise e

    print(f"🚀 Đang đóng gói diffusion_staging.tar sang Drive: {target_staging_tar}...")
    os.system(f'tar -cf "{target_staging_tar}" -C /content diffusion_staging')
    shutil.rmtree(staging_dir, ignore_errors=True)

    final_staging_mb = os.path.getsize(target_staging_tar) / (1024**2)
    assert final_staging_mb > 200, f"❌ Lỗi: File diffusion_staging.tar quá nhỏ ({final_staging_mb:.2f} MB), nghi vấn bị rỗng!"
    print(f"✅ HOÀN TẤT PHẦN 6.2: diffusion_staging.tar đã lưu thành công ({final_staging_mb:.2f} MB)!")

print()
print("=" * 70)
print("🎉 TOÀN BỘ BƯỚC 6 ĐÃ HOÀN TẤT XUẤT SẮC! CẢ 2 FILE TAR ĐÃ SẴN SÀNG TRÊN DRIVE!")
print("=" * 70)


## 7. Tổng Kiểm Tra & Kiểm Kê Kho Dữ Liệu Google Drive
Duyệt qua toàn bộ thư mục `Fatformer/` trên Drive, in ra báo cáo dung lượng thực tế và đánh giá trạng thái sẵn sàng huấn luyện.


In [ ]:
# 7. Kiểm kê toàn diện Google Drive
print("==========================================================================")
print("📊 BÁO CÁO TỔNG KHO DỮ LIỆU FATFORMER TRÊN GOOGLE DRIVE")
print("==========================================================================")

total_bytes = 0
for root, dirs, files in os.walk(DRIVE_ROOT):
    level = root.replace(DRIVE_ROOT, '').count(os.sep)
    indent = ' ' * 4 * level
    folder_name = os.path.basename(root) or "Fatformer"
    print(f"{indent}📂 {folder_name}/")
    subindent = ' ' * 4 * (level + 1)
    for f in sorted(files):
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total_bytes += size
        if size >= 1024**3:
            size_str = f"{size / (1024**3):.2f} GB"
        else:
            size_str = f"{size / (1024**2):.2f} MB"
        
        status = "✅ OK" if size > 10 * 1024 * 1024 else "⚠️ RỖNG/LỖI"
        print(f"{subindent}📄 {f:<34} [{size_str:>9}] {status}")

print("--------------------------------------------------------------------------")
total_gb = total_bytes / (1024**3)
print(f"🎯 TỔNG DUNG LƯỢNG ĐÃ NẠP: {total_gb:.2f} GB / 5.000 GB Drive ({total_gb/5000*100:.2f}%)")

# Đánh giá mức độ sẵn sàng
val_ok = os.path.exists(target_val_tar) and os.path.getsize(target_val_tar) > 500 * 1024 * 1024
train_ok = os.path.exists(target_train_tar) and os.path.getsize(target_train_tar) > 3 * 1024 * 1024 * 1024
gans_ok = os.path.exists(target_gans_tar) and os.path.getsize(target_gans_tar) > 8 * 1024 * 1024 * 1024
diff_ok = os.path.exists(target_diff_tar) and os.path.getsize(target_diff_tar) > 1 * 1024 * 1024 * 1024
staging_ok = os.path.exists(target_staging_tar) and os.path.getsize(target_staging_tar) > 10 * 1024 * 1024
ckpt_ok = os.path.exists(vit_path) and os.path.exists(ckpt_path)

if val_ok and train_ok and gans_ok and diff_ok and staging_ok and ckpt_ok:
    print("🚀 SẴN SÀNG TOÀN DIỆN: Toàn bộ dữ liệu Train, Val, Test GANs, Test Diffusion, Staging và Checkpoint đã nạp 100%!")
else:
    print("⏳ TRẠNG THÁI: Một số tệp dữ liệu cần chạy lại các cell phía trên để hoàn tất.")
print("==========================================================================")


## 8. Kiểm Tra Tính Toàn Vẹn File .TAR (Smoke Test)
Kiểm tra cấu trúc header của các file `.tar` trên Drive bằng lệnh `tar -tvf` (chỉ đọc mục lục không cần giải nén) để chứng minh 100% file không bị lỗi, không bị hỏng cấu trúc và sẵn sàng giải nén siêu tốc khi chạy Pipeline.


In [ ]:
tar_files = [
    ("progan_val.tar", os.path.join(DIR_DATASETS, "progan_val.tar")),
    ("progan_train.tar", os.path.join(DIR_DATASETS, "progan_train.tar")),
    ("test_benchmark_gans.tar", os.path.join(DIR_DATASETS, "test_benchmark_gans.tar")),
]

print("🔍 Đang kiểm tra tính hợp lệ của các file tar trên Drive...")
for name, p in tar_files:
    if os.path.exists(p) and os.path.getsize(p) > 10 * 1024 * 1024:
        print(f"""
--- Kiểm tra {name} ({os.path.getsize(p)/(1024**2):.1f} MB) ---""")
        !tar -tvf "{p}" | head -n 6
        print(f"  👉 Cấu trúc file {name}: HỢP LỆ VÀ NGUYÊN VẸN!")
    else:
        print(f"""
--- {name}: Chưa có hoặc file rỗng (< 10 MB).""")
